## Imports

In [162]:
import numpy as np
import scipy
import sklearn
import pandas as pd
import matplotlib as mpl
from matplotlib import pyplot as pyplot
import json
import os

## Load the data to visualize it

In [21]:
all_data_matrix = pd.read_csv('../Data/raw_data/all_subjects.csv')
subject_1 = all_data_matrix[all_data_matrix['subject']=='A2TQNX64349OZ9:3WSELTNVR4AZUVYAYCSWZIQW0J4ATB']

In [22]:
with open('../Data/raw_data/problems/prb48146_16.json') as prb48146_16:
    problem_1 = json.load(prb48146_16)
    print(problem_1)

{'id': 'prb48146', 'cars': [{'length': 2, 'id': '0', 'orientation': 'vertical', 'position': 16, 'player': False}, {'length': 2, 'id': '2', 'orientation': 'vertical', 'position': 15, 'player': False}, {'length': 2, 'id': '4', 'orientation': 'vertical', 'position': 8, 'player': False}, {'length': 3, 'id': '7', 'orientation': 'vertical', 'position': 17, 'player': False}, {'length': 2, 'id': '6', 'orientation': 'vertical', 'position': 0, 'player': False}, {'length': 3, 'id': '1', 'orientation': 'horizontal', 'position': 3, 'player': False}, {'length': 2, 'id': '3', 'orientation': 'horizontal', 'position': 1, 'player': False}, {'length': 2, 'id': 'r', 'orientation': 'horizontal', 'position': 12, 'player': True}, {'length': 3, 'id': '5', 'orientation': 'horizontal', 'position': 25, 'player': False}]}


## Creating a board/problem object to contain the information in the json file

In [ ]:
class move:
    def __init__(self,id,move,dir):
        self.id = id
        self.move = move
        self.dir = dir
    
    def unpack(self):
        return self.id,self.move,self.dir

class problem:
    def __init__(self,json_file):
        with open(json_file) as file:
            problem = json.load(file)
        self.id = problem['id']
        self._cars_tmp = problem['cars']
        self.board = np.zeros(36)
        self.board_2d = self.board.reshape(6,6)
        self._cars = {}
        self.history = []
        for i in self._cars_tmp:
            id = i['id']
            if id != 'r':
                id = str(int(id)+1)
            else:
                id = '-1'
            i_orient = i['orientation']
            i_pos = i['position']
            i_len = i['length']
            if i_orient == 'horizontal':
                sq_occ = np.arange(i_pos,i_pos+i_len)
            elif i_orient == 'vertical':
                sq_occ = np.arange(i_pos,i_pos+6*i_len,6)
            self.board[sq_occ] = int(id)

            self._cars[id] = {
                'orientation': i_orient,
                'position': i_pos,
                'length': i_len,
                'sq_occ': sq_occ}
    
    def display_board_2d(self):
        self.board_2d = self.board.reshape(6,6)
        print('-------------------------')
        for row in self.board_2d:
            print('|',end = '')
            print(*(' ' + str(int(i)) + ' ' if i > -1 else ' r ' for i in row),end = '')
            print('|',end = '\n')
        print('-------------------------')
    
    '''def legal_moves(self):
        self.board_2d = self.board.reshape(6,6)
        self.legal_moves = []
        for j in self._cars:
            j_orient = j['orientation']
            j_pos = j['position']
            j_len = j['length']
            if j_orient == 'horizontal':
                row = j_pos // 6
            if j_orient == 'vertical':
                col = j_pos % 6'''

    def hyp_move(self,move_obj):
        id,move,move_dir = move_obj.unpack()
        car = self._cars[id]
        sq_occ = car['sq_occ']
        sq_change = move
        if move_dir == 'vertical':
            sq_change = move * 6
        new_occ = sq_occ + sq_change
        return new_occ
    
    def check_legal(self,move_obj):
        id,move,move_dir = move_obj.unpack()
        car = self._cars[id]
        sq_occ = car['sq_occ']
        legal = True
        new_occ = self.hyp_move(move_obj)
        if move_dir != car['orientation']:
            #print('Illegal - car orientation and move direction conflict')
            legal = False
        if move_dir == 'horizontal' and np.any(new_occ // 6 != sq_occ // 6):
            #print('Illegal - car went out of bounds')
            legal = False
        elif np.any(new_occ < 0) or np.any(new_occ > 35):
            #print('Illegal - car went out of bounds')
            legal = False
        elif np.any((self.board[new_occ] != 0) == (self.board[new_occ] != int(id))):
            legal = 'Blocked'
        return legal,new_occ
        
        
    
    def blocked_by(self,move_obj):
        id,move,move_dir = move_obj.unpack()
        blocked,new_occ = self.check_legal(move_obj)
        if blocked == 'Blocked':
            if np.any((self.board[new_occ] != 0) == (self.board[new_occ] != int(id))):
                blocked_cars_id = np.unique(self.board[new_occ][(self.board[new_occ] != 0) == (self.board[new_occ] != int(id))])
                return [str(i) for i in blocked_cars_id]
            else:
                return new_occ
    
    def make_move(self,move_obj):
        id,move,move_dir = move_obj.unpack()
        car = self._cars[id]
        sq_occ = car['sq_occ']
        legal, new_occ = self.check_legal(move_obj)
        if legal == True:
            car['position'] = new_occ[0]
            car['sq_occ'] = new_occ
            self.board[sq_occ] = 0
            self.board[new_occ] = int(id)
            self.history.append(move_obj)
        elif legal == False:
            #print('Illegal move')
            return None
        else:
            return None
            #print('Illegal move - blocked by: ',*(str(int(float(i))) + ' ' for i in self.blocked_by(move_obj)))
    
    def undo_move(self):
        if not self.history:
            print('No moves to undo')
        else:
            last_move = self.history[-1]
            id,move_qt,move_dir = last_move.unpack()
            self.make_move(move(id,-move_qt,move_dir))
            self.history.pop()
            self.history.pop()
    
    def get_cars(self):
        return self._cars

In [170]:
dir_str = '../Data/raw_data/problems'
directory = os.fsencode(dir_str)
problem_list = {}
for file in os.listdir(directory):
    filename = os.fsdecode(file)
    key,extra = filename.split('.')
    unpacked_problem = problem(dir_str + '/' + str(filename))
    problem_list[key] = unpacked_problem

In [173]:
test_board = problem_list['prb46639_16']

In [174]:
test_board.display_board_2d()

-------------------------
| 6   0   0   0   1   1 |
| 6   7   7   7   0   0 |
| r   r   0   0   5   0 |
| 0   2   2   2   5   0 |
| 0   0   3   4   5   0 |
| 0   0   3   4   8   8 |
-------------------------


## Making AND/OR Tree